The object that we wish to compute: $\text{Re} D_m \bar{P}_n D_r (G_3)_{pqu}D_{s} (G_3)_{p'q'u'}$, or, for simplicity

$$
D_m \bar{P}_n D_r G_3 D_{s} G_3
$$

In [1]:
import sympy as sp

m,n,r,s = sp.symbols('m n r s', real=True)
tau1 = sp.Function('tau_1', real=True)(m, n, r, s)
tau2 = sp.Function('tau_2', real=True, positive=True)(m, n, r, s)
tau = tau1 + sp.I*tau2
taub = tau1 - sp.I*tau2

H3 = sp.Function('H3', real=True)(m, n, r, s)
F3 = sp.Function('F3', real=True)(m, n, r, s)
G3  = tau2**sp.Rational(-1,2) * (F3 - (tau1 + sp.I*tau2)*H3)

H3_sym = sp.Symbol('H_{3}')
F3_sym = sp.Symbol('F_{3}')

HF_subs = {
    H3: H3_sym,
    F3: F3_sym,
}

In [2]:
Pb_n = -sp.Rational(1, 2) * sp.I / tau2 * sp.diff(taub, n)
Q_r = -sp.Rational(1, 2) / tau2 * sp.diff(tau1, r)
Q_s = -sp.Rational(1, 2) / tau2 * sp.diff(tau1, s)
Q_m = -sp.Rational(1, 2) / tau2 * sp.diff(tau1, m)

In [3]:
D_r_G_3 = sp.diff(G3, r) - sp.I * Q_r * G3
D_s_G_3 = sp.diff(G3, s) - sp.I * Q_s * G3

In [4]:
from itertools import product as iterprod

coords = [m, n, r, s]
coord_names = {m: 'm', n: 'n', r: 'r', s: 's'}
funcs = [tau1, tau2, H3, F3]
func_names = {tau1: r'\tau_1', tau2: r'\tau_2', H3: r'H_3', F3: r'F_3'}

# Function symbols
T_func_syms = {}
T_func_subs = []
for f in funcs:
    sym = sp.Symbol(func_names[f])
    T_func_syms[f] = sym
    T_func_subs.append((f, sym))

# First derivative symbols
T_d1_syms = {}
T_deriv1_subs = []
for f, xi in iterprod(funcs, coords):
    deriv = sp.diff(f, xi)
    if deriv == 0:
        continue
    name = rf'\nabla_{{{coord_names[xi]}}} {func_names[f]}'
    sym = sp.Symbol(name)
    T_d1_syms[(f, xi)] = sym
    T_deriv1_subs.append((deriv, sym))

# Second derivative symbols
T_d2_syms = {}
T_deriv2_subs = []
for f, (xi, xj) in iterprod(funcs, iterprod(coords, coords)):
    deriv = sp.diff(f, xi, xj)
    if deriv == 0:
        continue
    key = (f, tuple(sorted([xi, xj], key=lambda x: coord_names[x])))
    if key not in T_d2_syms:
        ci, cj = coord_names[key[1][0]], coord_names[key[1][1]]
        name = rf'\nabla_{{{ci}}} \nabla_{{{cj}}} {func_names[f]}'
        T_d2_syms[key] = sp.Symbol(name)
    T_deriv2_subs.append((deriv, T_d2_syms[key]))

print(f"Function subs: {len(T_func_subs)}")
print(f"1st deriv subs: {len(T_deriv1_subs)}")
print(f"2nd deriv subs: {len(T_deriv2_subs)}")


Function subs: 4
1st deriv subs: 16
2nd deriv subs: 64


In [5]:
D_r_G_3.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs)

I*\nabla_{r} \tau_1*(F_3 - H_3*(\tau_1 + I*\tau_2))/(2*\tau_2**(3/2)) - \nabla_{r} \tau_2*(F_3 - H_3*(\tau_1 + I*\tau_2))/(2*\tau_2**(3/2)) + (-H_3*(\nabla_{r} \tau_1 + I*\nabla_{r} \tau_2) + \nabla_{r} F_3 - \nabla_{r} H_3*(\tau_1 + I*\tau_2))/sqrt(\tau_2)

In [6]:
D_r_G_3 = D_r_G_3.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)
D_s_G_3 = D_s_G_3.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)

In [7]:
D_r_G_3

I*\nabla_{r} \tau_1*(F_3 - H_3*(\tau_1 + I*\tau_2))/(2*\tau_2**(3/2)) - \nabla_{r} \tau_2*(F_3 - H_3*(\tau_1 + I*\tau_2))/(2*\tau_2**(3/2)) + (-H_3*(\nabla_{r} \tau_1 + I*\nabla_{r} \tau_2) + \nabla_{r} F_3 - \nabla_{r} H_3*(\tau_1 + I*\tau_2))/sqrt(\tau_2)

We find that
$$
D_r P_s = A + iB
$$
where
$$
A = \frac{\nabla_r \tau_"\nabla_s\tau_2}{2\tau_2^2} - \frac{\nabla_r\nabla_s\tau_2}{2\tau_2} - \frac{\nabla_r\tau_1\nabla_s\tau_1}{2\tau_2^2}
$$
$$\boxed{A = -\tau_2, R_{1r1s} + \tfrac{1}{2},g^{ab}R_{arbs}}$$
and
$$
B = \frac{\nabla_r\nabla_s\tau_1}{2\tau_2} - \frac{\nabla_r\tau_1\nabla_s\tau_2}{2\tau_2^2} - \frac{\nabla_r\tau_2\nabla_s\tau_1}{2\tau_2^2}
$$
$$\boxed{B = \tau_1, R_{1r1s} - \tfrac{1}{2}(R_{1r2s}+R_{2r1s})}$$

In [8]:
# Define Riemann tensor component symbols
tau1_sym = sp.Symbol(r'\tau_1')  # 	au_1
tau2_sym = sp.Symbol(r'\tau_2')  # 	au_2

# R_{1m1n}, R_{1m2n}, R_{2m1n}, R_{2m2n} (and same with r,s)
R_1m1n = sp.Symbol(r'R_{1m1n}')
R_1m2n = sp.Symbol(r'R_{1m2n}')
R_2m1n = sp.Symbol(r'R_{2m1n}')
R_2m2n = sp.Symbol(r'R_{2m2n}')
# R_trace_mn = sp.Symbol(r'g^{ab}R_{ambn}')  # trace
R_trace_mn = (tau1_sym**2 + tau2_sym**2)/tau2_sym * R_1m1n - tau1_sym / tau2_sym * (R_1m2n + R_2m1n) + 1/tau2_sym * R_2m2n

R_1r1s = sp.Symbol(r'R_{1r1s}')
R_1r2s = sp.Symbol(r'R_{1r2s}')
R_2r1s = sp.Symbol(r'R_{2r1s}')
R_2r2s = sp.Symbol(r'R_{2r2s}')
# R_trace_rs = sp.Symbol(r'g^{ab}R_{arbs}')  # trace
R_trace_rs = (tau1_sym**2 + tau2_sym**2)/tau2_sym * R_1r1s - tau1_sym / tau2_sym * (R_1r2s + R_2r1s) + 1/tau2_sym * R_2r2s

# D_r P_s = A_rs + i B_rs where:
# A_rs = -tau2 R_{1r1s} + (1/2) g^{ab} R_{arbs}
# B_rs = tau1 R_{1r1s} - (1/2)(R_{1r2s} + R_{2r1s})
A_rs = -tau2_sym * R_1r1s + sp.Rational(1,2) * R_trace_rs
B_rs = tau1_sym * R_1r1s - sp.Rational(1,2) * (R_1r2s + R_2r1s)

# D_m Pb_n = conj(D_m P_n) -> Re same, Im flipped
# D_m Pb_n = A_mn - i B_mn  (complex conjugate of D_m P_n)
A_mn = -tau2_sym * R_1m1n + sp.Rational(1,2) * R_trace_mn
B_mn = tau1_sym * R_1m1n - sp.Rational(1,2) * (R_1m2n + R_2m1n)

print('D_r P_s = A_rs + i B_rs')
print('A_rs =', A_rs)
print('B_rs =', B_rs)
print()
print('D_m Pb_n = A_mn - i B_mn')
print('A_mn =', A_mn)
print('B_mn =', B_mn)

D_m_Pb_n = A_mn - sp.I * B_mn

D_r P_s = A_rs + i B_rs
A_rs = -R_{1r1s}*\tau_2 + R_{1r1s}*(\tau_1**2 + \tau_2**2)/(2*\tau_2) + R_{2r2s}/(2*\tau_2) - \tau_1*(R_{1r2s} + R_{2r1s})/(2*\tau_2)
B_rs = R_{1r1s}*\tau_1 - R_{1r2s}/2 - R_{2r1s}/2

D_m Pb_n = A_mn - i B_mn
A_mn = -R_{1m1n}*\tau_2 + R_{1m1n}*(\tau_1**2 + \tau_2**2)/(2*\tau_2) + R_{2m2n}/(2*\tau_2) - \tau_1*(R_{1m2n} + R_{2m1n})/(2*\tau_2)
B_mn = R_{1m1n}*\tau_1 - R_{1m2n}/2 - R_{2m1n}/2


In [9]:
D_m_Pb_n

-R_{1m1n}*\tau_2 + R_{1m1n}*(\tau_1**2 + \tau_2**2)/(2*\tau_2) + R_{2m2n}/(2*\tau_2) - \tau_1*(R_{1m2n} + R_{2m1n})/(2*\tau_2) - I*(R_{1m1n}*\tau_1 - R_{1m2n}/2 - R_{2m1n}/2)

In [50]:
prod = D_m_Pb_n * D_r_G_3 * D_s_G_3
prod = sp.re(sp.expand(prod))
prod = prod.replace(lambda e: e.func == sp.re, lambda e: e.args[0])
prod = prod.replace(lambda e: e.func == sp.im, lambda e: sp.S.Zero)

# Covariantize the derivatives acting on $H_3$, $F_4$

In [51]:
# Metric
M = sp.Matrix([[1/tau2, tau1/tau2], [tau1/tau2, (tau1**2 + tau2**2)/tau2]])
M_inv = M.inv()

# Here we are defining Gamma^a_{m/n b}, in order (a,b)
Gamma_r = sp.Rational(1,2) * sp.simplify(M_inv * sp.diff(M,r))
Gamma_s = sp.Rational(1,2) * sp.simplify(M_inv * sp.diff(M,s))

In [52]:
nabla_r_F4_down_1_expr = sp.diff(H3, r) - Gamma_r[0,0]*H3 - Gamma_r[1,0]* F3
nabla_s_F4_down_1_expr = sp.diff(H3, s) - Gamma_s[0,0]*H3 - Gamma_s[1,0]* F3
nabla_r_F4_down_2_expr = sp.diff(F3, r) - Gamma_r[0,1]*H3 - Gamma_r[1,1]* F3
nabla_s_F4_down_2_expr = sp.diff(F3, s) - Gamma_s[0,1]*H3 - Gamma_s[1,1]* F3

nabla_r_F4_down_1_expr = nabla_r_F4_down_1_expr.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)
nabla_s_F4_down_1_expr = nabla_s_F4_down_1_expr.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)
nabla_r_F4_down_2_expr = nabla_r_F4_down_2_expr.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)
nabla_s_F4_down_2_expr = nabla_s_F4_down_2_expr.subs(T_deriv2_subs).subs(T_deriv1_subs).subs(T_func_subs).subs(HF_subs)

nabla_r_F4_down_1_sym = sp.Symbol(r'\nabla_{r} (F_4)_{1}')
nabla_s_F4_down_1_sym = sp.Symbol(r'\nabla_{s} (F_4)_{1}')
nabla_r_F4_down_2_sym = sp.Symbol(r'\nabla_{r} (F_4)_{2}')
nabla_s_F4_down_2_sym = sp.Symbol(r'\nabla_{s} (F_4)_{2}')

nabla_r_F4_up_1_sym = sp.Symbol(r'\nabla_{r} (F_4)^{1}')
nabla_s_F4_up_1_sym = sp.Symbol(r'\nabla_{s} (F_4)^{1}')
nabla_r_F4_up_2_sym = sp.Symbol(r'\nabla_{r} (F_4)^{2}')
nabla_s_F4_up_2_sym = sp.Symbol(r'\nabla_{s} (F_4)^{2}')

F4_down_1_sym = sp.Symbol(r'(F_4)_{1}')
F4_down_2_sym = sp.Symbol(r'(F_4)_{2}')

In [53]:
nabla_r_F4_down_2_sym

\nabla_{r} (F_4)_{2}

In [54]:
covariantization_subs = {
    T_d1_syms[(H3, r)]: nabla_r_F4_down_1_sym - (nabla_r_F4_down_1_expr - T_d1_syms[(H3, r)]),
    T_d1_syms[(H3, s)]: nabla_s_F4_down_1_sym - (nabla_s_F4_down_1_expr - T_d1_syms[(H3, s)]),
    T_d1_syms[(F3, r)]: nabla_r_F4_down_2_sym - (nabla_r_F4_down_2_expr - T_d1_syms[(F3, r)]),
    T_d1_syms[(F3, s)]: nabla_s_F4_down_2_sym - (nabla_s_F4_down_2_expr - T_d1_syms[(F3, s)]),
}

F4_down_subs = {
    T_func_syms[H3]: F4_down_1_sym,
    T_func_syms[F3]: F4_down_2_sym,
}


In [55]:
prod = prod.subs(covariantization_subs)
prod = prod.subs(F4_down_subs)
prod = sp.expand(sp.simplify(prod))

In [56]:
prod

R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*\tau_1**4/(2*\tau_2**2) + R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*\tau_1**2 + R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*\tau_2**2/2 - R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}*\tau_1**3/(2*\tau_2**2) - R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}*\tau_1/2 - R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}*\tau_1**3/(2*\tau_2**2) - R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}*\tau_1/2 + R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}*\tau_1**2/(2*\tau_2**2) - R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}/2 - R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*\tau_1**3/(2*\tau_2**2) - R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*\tau_1/2 + R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}*\tau_1**2/(2*\tau_2**2) + R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}/2 + R_{1m2n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}*\tau_1**2/(2*\tau_2**2) + R_{1m2n}*\nabla_{r} (F_4)_{2}*\nabla_

In [63]:
tau1_s = T_func_syms[tau1]
tau2_s = T_func_syms[tau2]

g_11 = 1 / tau2_s
g_12 = tau1_s / tau2_s
g_22 = (tau1_s**2 + tau2_s**2) / tau2_s

ginv_11 = g_22
ginv_12 = -g_12
ginv_22 = g_11

ginv_11_sym = sp.Symbol(r'g^{11}')
ginv_12_sym = sp.Symbol(r'g^{12}')
ginv_22_sym = sp.Symbol(r'g^{22}')

epsilon_12_up = sp.Symbol(r'\epsilon^{12}')

simplification_subs = {
    tau1_s**4 / tau2_s**2: (ginv_11_sym**2 - (ginv_11**2 - tau1_s**4 / tau2_s**2)),
    - tau1_s**3 / tau2_s**2:   (ginv_11_sym*ginv_12_sym - (ginv_11 *ginv_12 + tau1_s**3 / tau2_s**2)),
    (R_1m2n) * (-tau1_s / tau2_s**2):  (R_1m2n) * (ginv_12_sym * ginv_22_sym),
    (R_2m1n) * (-tau1_s / tau2_s**2):  (R_2m1n) * (ginv_12_sym * ginv_22_sym),
    (R_1m2n) * tau1_s**2 / tau2_s**2:  (R_1m2n) * (ginv_11_sym*ginv_22_sym-1),
    (R_2m1n) * tau1_s**2 / tau2_s**2:  (R_2m1n) * (ginv_11_sym*ginv_22_sym-1),
    (R_2m2n * nabla_r_F4_down_2_sym * nabla_s_F4_down_2_sym) / tau2_s**2:  (R_2m2n * nabla_r_F4_down_2_sym * nabla_s_F4_down_2_sym) * (ginv_22_sym**2),
    (R_2m2n * nabla_r_F4_down_2_sym * nabla_s_F4_down_1_sym) * (-tau1_s / tau2_s**2):  (R_2m2n * nabla_r_F4_down_2_sym * nabla_s_F4_down_1_sym) * (ginv_12_sym * ginv_22_sym),
    (R_2m2n * nabla_r_F4_down_1_sym * nabla_s_F4_down_2_sym) * (-tau1_s / tau2_s**2):  (R_2m2n * nabla_r_F4_down_1_sym * nabla_s_F4_down_2_sym) * (ginv_12_sym * ginv_22_sym),
    (R_1m1n) * (tau1_s**2 / tau2_s**2):(R_1m1n) * (ginv_12_sym * ginv_12_sym),
    (R_2m2n) * (tau1_s**2 / tau2_s**2):(R_2m2n) * (ginv_12_sym * ginv_12_sym),
    # (R_2r2s) * (- tau1_s / tau2_s**2): (R_2r2s) * (ginv_12_sym * ginv_22_sym),
    # (R_2m2n) * (- tau1_s / tau2_s**2): (R_2m2n) * (ginv_12_sym * ginv_22_sym),
    # (R_2m2n) * 1/ tau2_s **2 : (R_2m2n) * ginv_22_sym**2,
    # (R_1m1n) * (tau1_s**2 / tau2_s**2): (R_1m1n) * (ginv_12_sym**2), 
}
prod = prod.subs(simplification_subs)

In [64]:
prod = sp.expand(sp.simplify(prod))

In [65]:
sp.collect(prod, [ginv_11_sym ** 2, ginv_11_sym * ginv_12_sym, ginv_12_sym**2, ginv_12_sym * ginv_22_sym, ginv_22_sym**2, epsilon_12_up**2]) 

R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}*g^{11}**2/2 - R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}/2 + R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}*g^{11}*g^{22}/2 + R_{1m2n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}*g^{11}*g^{22}/2 + R_{2m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}*g^{11}*g^{22}/2 + R_{2m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}*g^{11}*g^{22}/2 - R_{2m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}/2 + R_{2m2n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}*g^{22}**2/2 + g^{11}*g^{12}*(R_{1m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{2}/2 + R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{1}/2 + R_{1m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}/2 + R_{2m1n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}/2) + g^{12}**2*(R_{1m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}/2 + R_{2m2n}*\nabla_{r} (F_4)_{1}*\nabla_{s} (F_4)_{1}/2) + g^{12}*g^{22}*(R_{1m2n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}/2 + R_{2m1n}*\nabla_{r} (F_4)_{2}*\nabla_{s} (F_4)_{2}/2 + R

Final result:

$$\boxed{\text{Re}\bigl[D_m\bar{P}_n D_r (G_3)_{pqu}, D_s (G_3)_{p'q'u'}\bigr] = \frac{1}{2}(g^{ac}g^{bd} - \epsilon^{ac}\epsilon^{bd}),R_{ambn},\hat{\nabla}_r(F_4)_{pquc},\hat{\nabla}_s(F_4)_{p'q'u'd}}$$


Taking into account the definition $DP_{mpnq} \equiv g_{[m[n} D_{p]} P_{q]}$ antisymmetrized with normalization 1, and the fact that 

$$
L_{4-pt} \sim \alpha f_0(\tau,\bar\tau)(t_8t_8-\tfrac14 \epsilon_8 \epsilon_8) 24 \text{Re}\left[R ( D\bar{P}(DG_3)^2)\right]
$$

from here, the contribution is

$$
12R(t_8t_8-\tfrac14 \epsilon_8 \epsilon_8)(g^{ac}g^{bd} - \epsilon^{ac}\epsilon^{bd})g_{m' n'}R_{ambn}\hat{\nabla}_r(F_4)_{pquc},\hat{\nabla}_s(F_4)_{p'q'u'd}
$$